In [7]:
%pip install -U pandas pyarrow fastparquet dask[dataframe]


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import glob

# --- 1. INITIAL SETUP ---

DATA_PATH = r"..\raw\yellow_tripdata_2022-*.parquet"

# Analysis Parameters
Z_SCORE_THRESHOLD = 3 

# Required Column Names
COL_DATETIME = 'tpep_pickup_datetime'
COL_DURATION = 'trip_duration_minutes'  # minutes
COL_SPEED = 'trip_speed_mph'             # mph

# ---------------------------------------------

def load_data(path: str) -> pd.DataFrame:
    """Load raw parquet files and create required trip-level features."""
    
    files = glob.glob(path)
    if len(files) == 0:
        raise FileNotFoundError("No raw parquet files found")

    print(f"Loading {len(files)} raw parquet files...")

    df = pd.concat(
        [pd.read_parquet(f, engine="fastparquet") for f in files],
        ignore_index=True
    )

    # Convert datetime
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
    df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

    # Trip duration (minutes)
    df[COL_DURATION] = (
        (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime'])
        .dt.total_seconds() / 60
    )

    # Trip speed (mph)
    df[COL_SPEED] = df['trip_distance'] / (df[COL_DURATION] / 60)

    # Basic cleaning (minimum needed for Z-score)
    df = df[
        (df[COL_DURATION] > 0) &
        (df[COL_DURATION] < 300) &
        (df[COL_SPEED] > 0) &
        (df[COL_SPEED] < 120)
    ]

    return df


def calculate_contextual_z_score(df: pd.DataFrame, analyze_col: str, z_threshold: float) -> pd.DataFrame:
    """
    Calculates the Z-score relative to the 'Hour of Day x Day of Week' context.
    """

    df['day_of_week'] = df[COL_DATETIME].dt.dayofweek
    df['hour'] = df[COL_DATETIME].dt.hour

    group_keys = ['day_of_week', 'hour']

    stats = df.groupby(group_keys)[analyze_col].agg(['mean', 'std'])
    stats.columns = [f'{analyze_col}_mean', f'{analyze_col}_std']

    df = df.merge(stats, on=group_keys, how='left')

    col_z = f'Z_{analyze_col}'
    col_outlier = f'Outlier_{analyze_col}'

    df[col_z] = (
        (df[analyze_col] - df[f'{analyze_col}_mean']) /
        df[f'{analyze_col}_std'].replace(0, 1)
    )

    df[col_outlier] = abs(df[col_z]) > z_threshold

    return df